In [4]:
from torch.utils.data import random_split
import torchvision.models.segmentation as segmentation
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import OxfordIIITPet


In [5]:
try: # disable certificate verification, needed on MacOS
    import ssl
    ssl._create_default_https_context = ssl._create_unverified_context
except ImportError:
    pass  # SSL module not available, skipping workaround

class SegmentationTransform:
    def __init__(self, size=(256, 256)):
        self.size = size
        self.image_transform = transforms.Compose([
            transforms.Resize(self.size),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],  # ImageNet stats
                                 std=[0.229, 0.224, 0.225])
        ])
        self.mask_transform = transforms.Compose([
            transforms.Resize(self.size, interpolation=transforms.InterpolationMode.NEAREST),
            transforms.PILToTensor()
        ])

    def __call__(self, image, target):
        image = self.image_transform(image)
        target = self.mask_transform(target).squeeze(0).long() - 1  # [1, H, W] → [H, W], label remap
        return image, target

train_dataset = OxfordIIITPet(
    root='.',
    target_types='segmentation',
    download=True,
    transforms=SegmentationTransform(size=(256, 256))
)

batch_size = 32  # TODO: change to your needs

train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_subset, val_subset = random_split(train_dataset, [train_size, val_size])
train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False)

def load_images(filename: str, image_shape: tuple = (3, 256, 256)):
    df = pd.read_csv(filename)

    images = []
    for _, row in df.iterrows():
        flat = eval(row['image'])  # Convert stringified list back to list
        tensor = torch.tensor(flat, dtype=torch.float32).reshape(image_shape)
        images.append(tensor)

    return torch.stack(images)  # [B, C, H, W]

We split our training data to evaluate the model and implement early stopping.

In [3]:
# Dataset and split
train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_subset, val_subset = random_split(train_dataset, [train_size, val_size])

# Create loaders
train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False)



We are using DeepLabV3 with a ResNet-50 backbone (deeplabv3_resnet50) pretrained model from torchvision.models

In [6]:
class SegmentationModel(nn.Module):
    def __init__(self, num_classes=3, freeze=True):
        super().__init__()
        # Load pretrained DeepLabV3
        #weights = segmentation.DeepLabV3_ResNet50_Weights.DEFAULT
        #self.model = segmentation.deeplabv3_resnet50(weights=weights)
        # Load the lightweight MobileNetV3 DeepLab model (because DeepLabV3 slow af - 2 epochs after 3 hours, this one took only 46 minutes)
        weights = segmentation.DeepLabV3_MobileNet_V3_Large_Weights.DEFAULT
        self.model = segmentation.deeplabv3_mobilenet_v3_large(weights=weights)

        # Freeze the backbone based on the parameter, as seen in NN_finetuning.ipynb
        if freeze:
            for param in self.model.backbone.parameters():
                param.requires_grad = False

        # Replace the classifier head for 3 classes (background, pet, edge)
        in_channels = self.model.classifier[4].in_channels
        self.model.classifier[4] = nn.Conv2d(in_channels, num_classes, kernel_size=1, stride=1)

        # Handle auxiliary classifier if present
        if self.model.aux_classifier is not None:
            in_channels_aux = self.model.aux_classifier[4].in_channels
            self.model.aux_classifier[4] = nn.Conv2d(in_channels_aux, num_classes, kernel_size=1, stride=1)

    def forward(self, x):
        # torchvision segmentation models output a dictionary. We want the main output.
        return self.model(x)['out']

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SegmentationModel(freeze=True).to(device)

Training and evaluation loop. We take our model and feed it data to build best model(takes a long time).

In [8]:
def dice_loss(pred, target, num_classes=3, smooth=1e-6):
    pred = torch.softmax(pred, dim=1)
    total = 0
    for cls in range(num_classes):
        p = pred[:, cls]
        t = (target == cls).float()
        intersection = (p * t).sum()
        total += 1 - (2 * intersection + smooth) / (p.sum() + t.sum() + smooth)
    return total / num_classes


In [9]:
# Your provided IoU function
def compute_iou(mask_true: np.ndarray, mask_pred: np.ndarray, num_classes: int = 3) -> float:
    """Computes mean Intersection over Union (mIoU) for multi-class masks."""
    ious = []

    for cls in range(num_classes):
        true_cls = mask_true == cls
        pred_cls = mask_pred == cls

        intersection = np.logical_and(true_cls, pred_cls).sum()
        union = np.logical_or(true_cls, pred_cls).sum()

        if union == 0:
            continue  # skip this class — not present in either pred or true

        iou = intersection / union
        ious.append(iou)

    if not ious:
        return 1.0  # If no classes are present in either mask, define mIoU as perfect

    return np.mean(ious)
#calculating section over union

# ── Train / Validate ──
def train(model, dataloader, optimizer, device):
    model.train()
    total_loss = 0
    criterion = nn.CrossEntropyLoss()
    for inputs, targets in dataloader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets) + dice_loss(outputs, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(dataloader)
    print(f"Train Loss: {avg_loss:.4f}")
    return avg_loss

def validate(model, dataloader, device):
    model.eval()
    total_loss = 0
    val_ious = []
    criterion = nn.CrossEntropyLoss()
    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets) + dice_loss(outputs, targets)
            total_loss += loss.item()
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            masks_np = targets.cpu().numpy()
            for i in range(preds.shape[0]):
                val_ious.append(compute_iou(masks_np[i], preds[i]))
    return total_loss / len(dataloader), np.mean(val_ious)

# ── Training: Phase 1 (frozen backbone) ──
best_iou = 0.0
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

for epoch in range(5):
    train_loss = train(model, train_loader, optimizer, device)
    val_loss, val_iou = validate(model, val_loader, device)
    scheduler.step(val_iou)
    print(f"[Phase 1] Epoch {epoch+1}/5 - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val mIoU: {val_iou:.4f}")
    if val_iou > best_iou:
        best_iou = val_iou
        torch.save(model.state_dict(), 'best_model.pth')
        print("Best model saved!")

# ── Training: Phase 2 (unfreeze backbone) ──
for param in model.model.backbone.parameters():
    param.requires_grad = True
optimizer = optim.Adam(model.parameters(), lr=5e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

for epoch in range(5):
    train_loss = train(model, train_loader, optimizer, device)
    val_loss, val_iou = validate(model, val_loader, device)
    scheduler.step(val_iou)
    print(f"[Phase 2] Epoch {epoch+1}/5 - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val mIoU: {val_iou:.4f}")
    if val_iou > best_iou:
        best_iou = val_iou
        torch.save(model.state_dict(), 'best_model.pth')
        print("Best model saved!")

Train Loss: 0.5734
[Phase 1] Epoch 1/5 - Train Loss: 0.5734, Val Loss: 0.4983, Val mIoU: 0.7104
Best model saved!
Train Loss: 0.4814
[Phase 1] Epoch 2/5 - Train Loss: 0.4814, Val Loss: 0.5124, Val mIoU: 0.7038
Train Loss: 0.4578
[Phase 1] Epoch 3/5 - Train Loss: 0.4578, Val Loss: 0.4949, Val mIoU: 0.7109
Best model saved!
Train Loss: 0.4298
[Phase 1] Epoch 4/5 - Train Loss: 0.4298, Val Loss: 0.4879, Val mIoU: 0.7145
Best model saved!
Train Loss: 0.4138
[Phase 1] Epoch 5/5 - Train Loss: 0.4138, Val Loss: 0.4951, Val mIoU: 0.7085
Train Loss: 0.3760
[Phase 2] Epoch 1/5 - Train Loss: 0.3760, Val Loss: 0.4493, Val mIoU: 0.7310
Best model saved!
Train Loss: 0.3510
[Phase 2] Epoch 2/5 - Train Loss: 0.3510, Val Loss: 0.4419, Val mIoU: 0.7354
Best model saved!
Train Loss: 0.3383
[Phase 2] Epoch 3/5 - Train Loss: 0.3383, Val Loss: 0.4387, Val mIoU: 0.7383
Best model saved!
Train Loss: 0.3270
[Phase 2] Epoch 4/5 - Train Loss: 0.3270, Val Loss: 0.4376, Val mIoU: 0.7393
Best model saved!
Train Loss

In [10]:
test_path = r"C:\Users\Richard\Desktop\daša škola\4. ročník\data science 2\ukol2\test_images.csv"
df = pd.read_csv(test_path)
ids = df.iloc[:, 0].values

In [11]:
# 1. Load test data and best model
test_tensor = load_images(test_path).to(device)
model.load_state_dict(torch.load('best_model.pth'))
model.eval()

# 2. Predict
predictions = []
with torch.no_grad():
    outputs = model(test_tensor)
    # Shape becomes [Batch, 256, 256]
    preds = torch.argmax(outputs, dim=1).cpu().numpy()

    # 3. Flatten and stringify
    for i in range(preds.shape[0]):
        flat_mask = preds[i].flatten().tolist()
        predictions.append(str(flat_mask))

# 4. Save to CSV
submission_df = pd.DataFrame({
    'id': ids,
    'mask': predictions,
    'Usage': 'Public'
})
submission_df.to_csv('submission3.csv', index=False)
#print("Submission saved!")

In [ ]:
import os
print(os.getcwd())

In [ ]:
import numpy as np

img, mask = train_dataset[0]

mask_np = np.array(mask)
print(np.unique(mask_np))